In [0]:
from pyspark.sql import functions as F

df_bronze_ia = spark.table("bronze_ia")

acentos_origem  = "ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
acentos_destino = "AAAAAEEEEIIIIOOOOOUUUUCN"

# 1. Explode a lista de procedimentos mantendo o cabeçalho e o objeto 'paciente'
df_bronze_json_separado = (
    df_bronze_ia
    .select(
        F.col("arquivo_origem"),
        F.col("unidade_clinica"),
        F.col("status_auditoria_guia"),
        F.col("latencia_processamento_ms"),
        F.col("paciente"),
        F.posexplode_outer("procedimentos").alias("posicao", "proc")
    )
)

# 2. Traduz as chaves aninhadas para o padrão plano da sua Silver
df_bronze_plana = (
    df_bronze_json_separado
    .select(
        # Transforma "guia_001.png" em "guia_001"
        F.regexp_replace(F.lower(F.trim(F.col("arquivo_origem"))), r"\.(png|jpg|jpeg|pdf)$", "").alias("id_guia"),
        F.coalesce(F.col("proc.sequencial_item").cast("int"), F.col("posicao") + 1).alias("item"),
        
        # Extrai de dentro do objeto 'paciente'
        F.col("paciente.nome_paciente_cadastro").alias("nome_ia"),
        F.col("paciente.carteira_plano_cadastro").alias("carteirinha_ia"),
        
        # Extrai de dentro do objeto 'proc'
        F.col("proc.codigo_tuss_ocr_bruto").alias("codigo_ocr_bruto"),
        F.col("proc.codigo_tuss_corrigido").alias("codigo_pos_rag"),
        F.col("proc.descricao_ocr_bruta").alias("descricao_ia"),
        
        # Metadados operacionais úteis para a Gold (opcionais)
        F.col("unidade_clinica"),
        F.col("status_auditoria_guia"),
        F.col("proc.metodo_correcao").alias("metodo_correcao_rag"),
        F.col("latencia_processamento_ms")
    )
)

# 3. Aplica exatamente a mesma régua de padronização do gabarito
limpo_bruto = F.regexp_replace(F.upper(F.trim(F.col("codigo_ocr_bruto"))), r"[^0-9A-Z]", "")
limpo_rag   = F.regexp_replace(F.upper(F.trim(F.col("codigo_pos_rag"))), r"[^0-9A-Z]", "")

df_silver_ia = (
    df_bronze_plana
    .withColumn("nome_ia", F.translate(F.upper(F.trim(F.col("nome_ia"))), acentos_origem, acentos_destino))
    .withColumn("carteirinha_ia", F.regexp_replace(F.trim(F.col("carteirinha_ia")), r"[^0-9]", ""))
    .withColumn("codigo_ocr_bruto", F.nullif(F.regexp_replace(limpo_bruto, r"^00(.{8})$", "$1"), F.lit("")))
    .withColumn("codigo_pos_rag", F.nullif(F.regexp_replace(limpo_rag, r"^00(.{8})$", "$1"), F.lit("")))
    .withColumn(
        "descricao_ia",
        F.translate(
            F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("descricao_ia"), '"', ''), r"\s+", " "))),
            acentos_origem,
            acentos_destino
        )
    )
)

df_silver_ia.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ia")
display(df_silver_ia)

In [0]:
# ==================== CÉLULA 2: JOIN E ACURÁCIA ====================
from pyspark.sql import functions as F

df_gab_completo = spark.table("silver_gabarito").withColumn("item", F.col("item").cast("int"))
df_ia           = spark.table("silver_ia").withColumn("item", F.col("item").cast("int"))

# Filtra o gabarito apenas para as guias presentes no arquivo da IA
df_gab = df_gab_completo.join(df_ia.select("id_guia").distinct(), on="id_guia", how="left_semi")

df_silver_join = (
    df_gab.alias("gab")
    .join(df_ia.alias("ia"), on=["id_guia", "item"], how="left")
    .withColumn(
        "acerto_nome",
        F.when(F.coalesce(F.col("ia.nome_ia"), F.lit("")) == F.coalesce(F.col("gab.nome"), F.lit("")), 1).otherwise(0)
    )
    .withColumn(
        "acerto_carteirinha",
        F.when(
            F.col("gab.carteirinha").isNotNull() & (F.trim(F.col("gab.carteirinha")) != ""),
            F.when(F.coalesce(F.col("ia.carteirinha_ia"), F.lit("")) == F.col("gab.carteirinha"), 1).otherwise(0)
        )
    )
    .withColumn(
        "acerto_ocr_bruto",
        F.when(F.coalesce(F.col("ia.codigo_ocr_bruto"), F.lit("")) == F.coalesce(F.col("gab.codigo_tuss"), F.lit("")), 1).otherwise(0)
    )
   .withColumn(
        "acerto_pos_rag",
        F.when(
            F.col("gab.codigo_tuss").isNotNull() & (F.trim(F.col("gab.codigo_tuss")) != ""),
            F.when(F.coalesce(F.col("ia.codigo_pos_rag"), F.lit("")) == F.col("gab.codigo_tuss"), 1).otherwise(0)
        )
    )
    .withColumn(
        "acerto_descricao",
        F.when(F.coalesce(F.col("ia.descricao_ia"), F.lit("")) == F.coalesce(F.col("gab.descricao"), F.lit("")), 1).otherwise(0)
    )
    .select(
        "id_guia",
        "item",
        F.col("gab.nome").alias("nome_gabarito"),
        F.col("ia.nome_ia").alias("nome_ia"),
        F.col("gab.carteirinha").alias("carteirinha_gabarito"),
        F.col("ia.carteirinha_ia").alias("carteirinha_ia"),
        F.col("gab.codigo_tuss").alias("tuss_gabarito"),
        F.col("ia.codigo_ocr_bruto").alias("tuss_ocr_bruto"),
        F.col("ia.codigo_pos_rag").alias("tuss_pos_rag"),
        F.col("gab.descricao").alias("descricao_gabarito"),
        F.col("ia.descricao_ia").alias("descricao_ia"),
        "acerto_nome",
        "acerto_carteirinha",
        "acerto_ocr_bruto",
        "acerto_pos_rag",
        "acerto_descricao"
    )
)

display(df_silver_join)

# Acurácia de cabeçalho (1 por guia)
acuracia_nome_carteirinha = (
    df_silver_join.select("id_guia", "acerto_nome", "acerto_carteirinha")
    .dropDuplicates(["id_guia"])
    .agg(
        F.round(F.avg("acerto_nome") * 100, 2).alias("acc_nome_pct"),
        F.round(F.avg("acerto_carteirinha") * 100, 2).alias("acc_carteirinha_pct")
    )
)

# Acurácia dos itens/exames (TUSS bruto, TUSS pós-RAG e descrição)
acuracia_itens = (
    df_silver_join
    .agg(
        F.round(F.avg("acerto_ocr_bruto") * 100, 2).alias("acc_tuss_bruto_pct"),
        F.round(F.avg("acerto_pos_rag") * 100, 2).alias("acc_tuss_pos_rag_pct"),
        F.round(F.avg("acerto_descricao") * 100, 2).alias("acc_descricao_pct")
    )
)



display(acuracia_nome_carteirinha)
display(acuracia_itens)